In [40]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from cryptosense.data import load_processed
from cryptosense.features import add_features

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score


DATA_DIR = Path.cwd().parent / "data"
df = add_features(load_processed("BTC/USDT", DATA_DIR))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
up=(df["dist_sma_100"]>0) & (df["mom_20"]>0)
down=(df["dist_sma_100"]<0) & (df["mom_20"]<0)
labels=np.select([up, down], ["up", "down"], default="range")

valid = df[["dist_sma_100", "mom_20"]].notna().all(axis=1)
df["trend"] = pd.Series(labels, index=df.index).where(valid)

In [42]:
df["trend"].value_counts(normalize=True)

trend
up       0.389658
down     0.325639
range    0.284703
Name: proportion, dtype: float64

In [43]:
fig = px.scatter(df, x=df.index, y="close", color="trend", log_y=True,
                 title="BTC/USDT trend regimes")
fig.show()

In [44]:
df.groupby("trend")["er_20"].mean()

trend
down     0.258126
range    0.146220
up       0.328116
Name: er_20, dtype: float64

In [45]:
run_id = (df["trend"] != df["trend"].shift()).cumsum().rename("run")
runs = df.groupby(run_id)["trend"].agg(["first", "size"])
runs.columns = ["trend", "days"]
runs.dropna().groupby("trend")["days"].describe()

,count,mean,std,min,25%,50%,75%,max
trend,,,,,,,,
down,122.0,8.672131,11.900026,1.0,1.0,3.0,9.75,54.0
range,212.0,4.363208,5.306275,1.0,1.0,2.0,5.00,30.0
up,100.0,12.660000,16.677100,1.0,2.0,5.0,19.25,80.0


In [46]:
labelled = runs.dropna()
long_days = labelled.loc[labelled["days"] >= 10, "days"].sum()
print(long_days / labelled["days"].sum())

0.7226839027393044


### Trend regime stability

**Observation:** Half of all trend-regime runs last 5 days or less for "up", 3 for "down" and 2 for "range", and a quarter last a single day. However, 72% of labelled days fall in runs of 10 days or more.

**Interpretation:** The labels are stable during clear trends but flicker around regime changes, where 20-day momentum hovers near zero. A remedy (a neutral band or a minimum duration) should be chosen for that reason, not by testing which version produces better returns.

In [47]:
vol_ref = df["vol_20"].rolling(365).median()

labels = np.where(df["vol_20"] > vol_ref, "high", "low")
valid = df["vol_20"].notna() & vol_ref.notna()
df["vol_regime"] = pd.Series(labels, index=df.index).where(valid)
df["vol_regime"].value_counts(dropna=False)
df["vol_regime"].value_counts(normalize=True)

vol_regime
low     0.542173
high    0.457827
Name: proportion, dtype: float64

### Volatility regime stability

**Observation:** Volatility regimes have median runs of 10 days ("high") and 7 days ("low"), with means of 18 and 21 days, compared with medians of 2–5 days for trend regimes. There are 149 volatility runs versus 434 trend runs. "Low" runs have the longer tail (75th percentile 36 days, maximum 122).

**Interpretation:** This was predicted before checking: `vol_20` changes slowly (consecutive values share 19 of 20 days) and volatility clusters, so volatility regimes persist for weeks. This makes recent volatility a strong baseline forecast for Phase 4. Some flicker remains when `vol_20` sits near its reference median.

In [48]:
run_id = (df["vol_regime"] != df["vol_regime"].shift()).cumsum().rename("run")
runs_vol = df.groupby(run_id)["vol_regime"].agg(["first", "size"])
runs_vol.columns = ["vol_regime", "days"]
runs_vol.dropna().groupby("vol_regime")["days"].describe()

,count,mean,std,min,25%,50%,75%,max
vol_regime,,,,,,,,
high,74.0,18.337838,22.798268,1.0,4.0,10.0,22.25,115.0
low,75.0,21.426667,25.567173,1.0,2.0,7.0,36.00,122.0


### Trend versus volatility

**Observation:** Across all labelled days, 46% are high-volatility. The share is 51% on "down" days, 47% on "up" days and 38% on "range" days.

**Interpretation:** Down days are slightly more often high-volatility than up days, consistent with the stock-market pattern of more turbulent declines, but the 4-point difference is small and not established. "Range" days are clearly calmer, which, together with their low efficiency ratio (0.15), suggests they are choppy but quiet periods. The volatility label is relative to the past year, not an absolute measure.

In [49]:
pd.crosstab(df["trend"], df["vol_regime"], normalize="index")

vol_regime,high,low
trend,,
down,0.512009,0.487991
range,0.384706,0.615294
up,0.468280,0.531720


### Trend versus volatility

**Observation:** Across all labelled days, 46% are high-volatility. The share is 51% on "down" days, 47% on "up" days and 38% on "range" days.

**Interpretation:** Down days are slightly more often high-volatility than up days, consistent with the stock-market pattern of more turbulent declines, but the 4-point difference is small and not established. "Range" days are clearly calmer, which, together with their low efficiency ratio (0.15), suggests they are choppy but quiet periods. The volatility label is relative to the past year, not an absolute measure.

In [50]:
#kmeans model 
FEATURES = ["mom_20", "dist_sma_100", "vol_20", "er_20"]
X = df[FEATURES].dropna()

train = X.loc[:"2023-12-31"]
test = X.loc["2024-01-01":]


In [51]:
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train)   # learn mean/std from train only
test_scaled = scaler.transform(test)         # apply train's mean/std, learn nothing

summary = pd.DataFrame({
    "train_mean": train_scaled.mean(axis=0),
    "train_std": train_scaled.std(axis=0),
    "test_mean": test_scaled.mean(axis=0),
    "test_std": test_scaled.std(axis=0),
}, index=FEATURES)
summary.round(2)

,train_mean,train_std,test_mean,test_std
mom_20,-0.0,1.0,-0.02,0.60
dist_sma_100,-0.0,1.0,-0.07,0.53
vol_20,0.0,1.0,-0.68,0.48
er_20,0.0,1.0,-0.13,0.95


In [52]:
rows = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=0)
    labels = model.fit_predict(train_scaled)
    rows.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(train_scaled, labels),
    })

pd.DataFrame(rows).round(3)

,k,inertia,silhouette
0,2,6218.159,0.367
1,3,4589.592,0.363
2,4,3909.795,0.353
3,5,3516.863,0.352
4,6,2895.760,0.361
5,7,2549.932,0.290
6,8,2318.484,0.306


In [53]:
for k in [3, 4]:
    model = KMeans(n_clusters=k, random_state=0).fit(train_scaled)

    centres = pd.DataFrame(
        scaler.inverse_transform(model.cluster_centers_),
        columns=FEATURES,
    )
    centres["share_of_days"] = pd.Series(model.labels_).value_counts(normalize=True).sort_index()

    print(f"k = {k}")
    print(centres.round(3))
    print()

k = 3
   mom_20  dist_sma_100  vol_20  er_20  share_of_days
0   0.266         0.392   0.036  0.497          0.203
1   0.007         0.025   0.029  0.153          0.618
2  -0.239        -0.194   0.051  0.366          0.179

k = 4
   mom_20  dist_sma_100  vol_20  er_20  share_of_days
0   0.017         0.023   0.028  0.142          0.566
1  -0.214        -0.205   0.039  0.389          0.180
2  -0.177         0.118   0.081  0.186          0.052
3   0.266         0.386   0.036  0.501          0.202



In [54]:
model4 = KMeans(n_clusters=4, random_state=0).fit(train_scaled)
shock_days = train.index[model4.labels_ == 2]
print(len(shock_days))
print(pd.Series(shock_days.year).value_counts().sort_index())

115
date
2017    10
2018    53
2019    19
2020    20
2021    13
Name: count, dtype: int64


In [55]:
model3 = KMeans(n_clusters=3, random_state=0).fit(train_scaled)
test_labels = model3.predict(test_scaled)

pd.DataFrame({
    "train_share": pd.Series(model3.labels_).value_counts(normalize=True).sort_index(),
    "test_share": pd.Series(test_labels).value_counts(normalize=True).sort_index(),
}).round(3)

,train_share,test_share
0,0.203,0.133
1,0.618,0.798
2,0.179,0.069


In [56]:
pd.crosstab(df.loc[train.index, "trend"], model3.labels_)

col_0,0,1,2
trend,,,
down,0,374,354
range,19,595,44
up,434,409,0


In [57]:
from sklearn.metrics import adjusted_rand_score

base = KMeans(n_clusters=3, random_state=0).fit_predict(train_scaled)
for seed in range(1, 6):
    other = KMeans(n_clusters=3, random_state=seed).fit_predict(train_scaled)
    print(seed, round(adjusted_rand_score(base, other), 3))

1 0.999
2 0.949
3 0.946
4 0.95
5 0.967


### K-means clustering: findings

**Observation:** With four standardized features (`mom_20`, `dist_sma_100`, `vol_20`, `er_20`), fitted on 2017–2023, silhouette scores were 0.35–0.37 for k = 2 to 6, about double the 0.17–0.19 obtained from random features. At k = 3, the clusters are a strong clean uptrend (20% of days), a quiet choppy state (62%) and a volatile downtrend (18%, annualized volatility about 97% versus 69% for the uptrend). No rule-"up" day fell in the downtrend cluster and no rule-"down" day in the uptrend cluster; rule "up" and "down" days split roughly evenly between their strong cluster and the neutral one. Results were stable across random seeds (ARI 0.95–0.999). On 2024+ test data, the neutral cluster grew to 80% of days. A fourth cluster at k = 4 (violent drops above the long-term trend, 5% of days) occurred mainly in 2017–2021 and not at all in 2022–2023.

**Interpretation:** The data has real but moderate structure. K-means rediscovered the rule dimensions (direction and choppiness) and added one distinction, trend intensity, rather than revealing new regimes. Because `vol_20` is an absolute measure and BTC's volatility appears to have declined over time, the clusters partly encode the market's era, which explains the drift on test data. A relative volatility feature would be needed before relying on clustering.